# CellWiki Demo

CellWiki 是一个自维护的单细胞多组学知识库，采用 "LLM Wiki" 模式构建。

本 notebook 演示端到端的工作流程：
1. 加载配置与 Cell Ontology
2. 数据模型概览
3. PDF 论文文本提取
4. LLM 驱动的细胞类型知识抽取
5. 知识融合与 Wiki 重建
6. Wiki 查询
7. 细胞类型关系图可视化与分析

## 0. 环境准备

In [ ]:
import sys
from pathlib import Path

# 将 src/ 加入 path，使 cellwiki 包可导入
project_root = Path.cwd().resolve()
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

## 1. 配置与初始化

In [ ]:
from cellwiki.config import settings, ensure_dirs

ensure_dirs()

print("=== 项目目录结构 ===")
print(f"Project Root : {settings.project_root}")
print(f"Data Dir     : {settings.data_dir}")
print(f"Wiki Dir     : {settings.wiki_dir}")
print(f"Cell Types   : {settings.wiki_cell_types_dir}")
print(f"Extractions  : {settings.extractions_dir}")
print(f"Ontology     : {settings.cell_ontology_file}")

## 2. 数据模型

In [ ]:
from cellwiki.models import (
    Marker, MarkerType, FunctionalCharacteristic,
    PaperReference, CellTypeExtract, ExtractionResult,
    WikiCellType, MarkerGene, Tissue, Disease,
    Trajectory, TrajectoryState, EvidenceTier,
)

print("=== 数据模型 ===")
models = [Marker, MarkerType, FunctionalCharacteristic,
          PaperReference, CellTypeExtract, ExtractionResult,
          WikiCellType, MarkerGene, Tissue, Disease,
          Trajectory, TrajectoryState, EvidenceTier]
for m in models:
    print(f"  {m.__name__:25s} -> {len(m.model_fields) if hasattr(m, 'model_fields') else 'Enum'} fields")

### 构造一个示例论文引用

In [ ]:
from cellwiki.models import PaperReference

sample_paper = PaperReference(
    paper_id="demo_paper_001",
    title="Single-cell transcriptomic analysis reveals tumor microenvironment heterogeneity",
    doi="10.1234/demo.2025",
    year=2025,
)
print(sample_paper.model_dump_json(indent=2))

## 3. Cell Ontology 加载与 CL ID 解析

In [ ]:
from cellwiki.ontology import load_cell_ontology, resolve_cell_type_to_cl

# 加载本体（从缓存 OBO 文件）
ontology = load_cell_ontology()
print(f"Loaded {len(ontology.get('terms', {}))} ontology terms")
print(f"Loaded {len(ontology.get('synonym_map', {}))} synonym entries")

# 解析细胞类型名称 → CL ID
test_names = ["CD8+ T cell", "Macrophage", "B cell", "NK cell"]
for name in test_names:
    cl_id = resolve_cell_type_to_cl(name, ontology)
    print(f"  {name:20s} -> {cl_id}")

## 4. PDF 论文文本提取

In [ ]:
from cellwiki.extract import extract_pdf_to_text

# 使用项目自带的示例论文
pdf_path = settings.references_dir / "exmaple_paper_1.pdf"
print(f"PDF: {pdf_path}")
print(f"Exists: {pdf_path.exists()}")

text = extract_pdf_to_text(pdf_path)
print(f"\nExtracted {len(text)} characters")
print(f"\n--- 前 500 字符 ---")
print(text[:500])

## 5. LLM 驱动的细胞类型知识抽取

> 此步骤需要配置 OpenAI 兼容 API（`OPENAI_API_KEY` 和 `OPENAI_BASE_URL`）。如果尚未配置，此 cell 会跳过，直接进入后续环节。

In [ ]:
import os

has_api = bool(settings.openai_api_key)
print(f"OpenAI API configured: {has_api}")
print(f"Model: {settings.openai_model}")

if has_api and text:
    from cellwiki.llm_extract import extract_cell_types_from_paper, chunk_text
    
    # 将文本分块
    chunks = chunk_text(text)
    print(f"Split paper into {len(chunks)} chunks")
    
    # LLM 抽取
    extraction = extract_cell_types_from_paper(text, sample_paper)
    
    print(f"\n=== 抽取结果 ===")
    print(f"Extracted {len(extraction.cell_types)} cell types")
    for ct in extraction.cell_types:
        print(f"  - {ct.name} (CL: {ct.cl_id or 'unresolved'})")
        print(f"    Markers: {[m.gene_symbol for m in ct.markers]}")
        print(f"    Functions: {[f.description[:40] for f in ct.functions]}")
else:
    print("Skipping LLM extraction (no API key configured).")
    print("Set OPENAI_API_KEY and OPENAI_BASE_URL to enable this step.")

## 6. Wiki 现状浏览

In [ ]:
from pathlib import Path

wiki_cell_types_dir = settings.wiki_cell_types_dir
md_files = list(wiki_cell_types_dir.glob("*.md"))
print(f"Wiki 中共有 {len(md_files)} 个细胞类型页面\n")

# 随机展示 5 个页面
import random
for f in random.sample(md_files, min(5, len(md_files))):
    print(f"  {f.stem}")

### 查看一个完整的 Wiki 页面

In [ ]:
# 读取并展示 cd8_exhausted_t_cell 页面
page_path = wiki_cell_types_dir / "cd8_exhausted_t_cell.md"
with open(page_path) as f:
    content = f.read()

print(content)

## 7. 细胞类型关系图可视化

> 展示已有的关系图（如果存在 SVG）

In [ ]:
from IPython.display import SVG, display

svg_path = settings.wiki_dir / "graph.svg"
if svg_path.exists():
    display(SVG(filename=str(svg_path)))
else:
    # Try to generate the relationship graph from existing extractions
    try:
        from cellwiki.visualization import generate_relationship_graph
        print("Generating relationship graph...")
        generate_relationship_graph()
        if svg_path.exists():
            display(SVG(filename=str(svg_path)))
        else:
            print("No graph SVG found after generation. Run 'cellwiki graph' to generate it.")
    except Exception as e:
        print(f"Graph generation skipped: {e}")
        print("Run 'cellwiki graph' to generate the relationship visualization.")

## 8. 图分析：Hub 检测、社区发现、桥接节点

In [ ]:
try:
    from cellwiki.graph_analysis import CellWikiGraph, build_and_analyze
    
    # build_and_analyze returns (graph, insights) tuple
    graph, insights = build_and_analyze()
    
    print("\n=== 图统计 ===")
    print(f"  节点数: {insights.get('total_nodes', 0)}")
    print(f"  边数:   {insights.get('total_edges', 0)}")
    print(f"  连通分量: {insights.get('components', 0)}")
    
    if insights.get("top_hubs"):
        print("\n=== Top Hub 节点 (度中心性) ===")
        for h in insights["top_hubs"][:10]:
            print(f"  {h['node']:40s}  degree={h['degree']}")
    
    if insights.get("isolated_nodes"):
        print(f"\n=== 孤立节点 ({len(insights['isolated_nodes'])}) ===")
        for n in insights["isolated_nodes"][:10]:
            print(f"  - {n}")
    
    if insights.get("communities"):
        print("\n=== 社区发现 (Louvain) ===")
        for i, comm in enumerate(insights["communities"][:5]):
            members = comm.get("members", [])
            print(f"  Community {i}: {len(members)} nodes")
            for node in members[:5]:
                print(f"    - {node}")
    
    if insights.get("bridges"):
        print("\n=== 桥接节点 (Betweenness) ===")
        for h in insights["bridges"][:5]:
            print(f"  {h['node']:40s}  betweenness={h['betweenness']:.4f}")
except ImportError:
    print("graph_analysis requires networkx: pip install networkx python-louvain")
except Exception as e:
    print(f"Graph analysis failed: {e}")

## 9. Wiki 查询演示

> 从已有的 Wiki 页面中直接检索信息

In [ ]:
from cellwiki.wiki import _proper_title_case
from cellwiki.knowledge import load_all_extractions, merge_to_wiki

# 展示标题格式化
examples = ["cd8_exhausted_t_cell", "regulatory_t_cell", "m1_macrophage"]
for name in examples:
    print(f"  {name:30s} -> {_proper_title_case(name)}")

# 加载所有提取数据
extractions = load_all_extractions()
print(f"\nLoaded {len(extractions)} extraction results")

# 合并后的 Wiki 数据
wiki_types = merge_to_wiki(extractions)
print(f"Merged into {len(wiki_types)} canonical cell types")

## 10. 审计与健康检查

In [ ]:
from cellwiki.audit import run_audit

issues = run_audit()

print(f"=== 审计报告 ===")
print(f"总问题数: {len(issues)}")

# 按类别分组
by_type = {}
for issue in issues:
    t = issue.get("type", "unknown")
    by_type.setdefault(t, []).append(issue)

for t, items in by_type.items():
    print(f"\n  [{t}] {len(items)} issues")
    for item in items[:3]:
        desc = item.get("description", item.get("entity", ""))
        print(f"    - {desc}")
    if len(items) > 3:
        print(f"    ... and {len(items) - 3} more")

## 总结

本 notebook 演示了 CellWiki 的完整工作流：

| 步骤 | 模块 | 说明 |
|------|------|------|
| 配置 | `config.py` | Pydantic 设置，路径管理 |
| 本体 | `ontology.py` | Cell Ontology 加载与 CL ID 解析 |
| 提取 | `extract.py` | PDF 文本提取 |
| LLM | `llm_extract.py` | 细胞类型/Marker/功能抽取 |
| 融合 | `knowledge.py` | 多源知识合并去重 |
| Wiki | `wiki.py` | Markdown 页面生成 |
| 审计 | `audit.py` | 冲突检测、缺失 CL ID |
| 可视化 | `visualization.py` | 关系图 JSON/DOT/Mermaid/SVG |
| 图分析 | `graph_analysis.py` | Hub/社区/桥接节点分析 |

v2 LangGraph 子图（`ingest`, `query`, `lint`, `research`）可通过 CLI 命令 `cellwiki ingest` / `cellwiki query-v2` / `cellwiki lint-v2` 调用。